In [ ]:
import fiona
from shapely.geometry import shape

In [ ]:
import pyproj
# wgs84 = pyproj.Proj(proj='latlong', datum='WGS84')
# proj_daymet = pyproj.Proj('+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs')
wgs84 = pyproj.CRS.from_proj4("+proj=latlong +datum=WGS84")
proj_daymet = pyproj.CRS.from_proj4("+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 +x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs")

input_crs_daymet = True # input shp is in daymet crs, usually generate by watershed-workflow

In [ ]:
input_shapefile = "./naches/naches.shp"
output_xyz_file = "naches_wbd.xyz"

with fiona.open(input_shapefile, 'r') as source:
    # Assuming the shapefile contains only the Naches watershed polygon
    # Or you can iterate and filter by HUC if needed
    for feature in source:
        geom = shape(feature['geometry'])
        # Polygons have an exterior ring and potentially interior rings
        # We usually care about the exterior boundary for a watershed
        exterior_coords = geom.exterior.coords

        with open(output_xyz_file, 'w') as f:
            for x, y, *z in exterior_coords: # *z handles cases with or without Z
                if z:
                    f.write(f"{x} {y} {z[0]}\n")
                else:
                    f.write(f"{x} {y}\n")
        break # Assuming only one feature for the Naches watershed
print(f"Naches watershed boundary exported to {output_xyz_file}")

In [ ]:
input_shapefile = "./oakcreek/OakCreek_bounds.shp"
output_xyz_file = "oakcreek_wbd.xyz"

if input_crs_daymet:
    transformer = pyproj.Transformer.from_crs(proj_daymet, wgs84)

with fiona.open(input_shapefile, 'r') as source:
    # Assuming the shapefile contains only the Naches watershed polygon
    # Or you can iterate and filter by HUC if needed
    for feature in source:
        geom = shape(feature['geometry'])
        # Polygons have an exterior ring and potentially interior rings
        # We usually care about the exterior boundary for a watershed
        exterior_coords = geom.exterior.coords

        with open(output_xyz_file, 'w') as f:
            for x, y, *z in exterior_coords: # *z handles cases with or without Z
                if input_crs_daymet:
                    #x_wgs84, y_wgs84 = pyproj.transform(proj_daymet, wgs84, x, y)
                    x_wgs84, y_wgs84 = transformer.transform(x, y)
                    x = x_wgs84
                    y = y_wgs84
                    
                if z:
                    f.write(f"{x} {y} {z[0]}\n")
                else:
                    f.write(f"{x} {y}\n")
        break # Assuming only one feature for the Naches watershed
print(f"Oak Creek watershed boundary exported to {output_xyz_file}")